<a href="https://colab.research.google.com/github/guitorte/audio/blob/ccr-8be874fb-rdbsnf/vocal-lab/Vocal_Stem_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Vocal Stem Lab

A workbench for one vocal: pull it out of a song, split lead from backing, remove room reverb and noise, rebuild missing highs, turn the melody into MIDI, and cut it into phrases.

**One file is "active" at a time.** Each module works on it and passes its result to the next, so you can simply run the modules you want, top to bottom. The **Router** goes back to an earlier stage.

| Module | What it does | Engine |
|---|---|---|
| **3 · Analyse** | tempo, key, loudness of the song | Beat This, librosa, ffmpeg |
| **A · Separate** | vocal out of the song; lead out of the vocal | BS/Mel-RoFormer, Demucs |
| **B · De-reverb** | removes room sound | Mel-RoFormer |
| **C · Denoise** | removes hiss and background noise | Mel-RoFormer |
| **D · Upscale** | rebuilds highs above the point where they were cut, in stereo | AudioSR, or FlashSR (experimental) |
| **E · MIDI** | the sung melody as notes | GAME |
| **F · Chopper** | phrases as a sample pack | librosa |

Everything is copied to `MyDrive/audio/outputs/<project_name>/`. Your source file is never changed.

**How to use:** `Runtime ▸ Change runtime type ▸ T4 GPU`, then run cells 1 and 2, then any modules. The first use of each module downloads its model.

In [ ]:
#@title 1 · Connect Google Drive and open a project
project_name = "my-song"  #@param {type:"string"}
#@markdown Results are copied to `MyDrive/audio/outputs/<project_name>/`. Type another name and run this cell again to start a new project.

import os, subprocess, sys
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

REPO, BRANCH = "/content/audio", "ccr-8be874fb-rdbsnf"
if not os.path.isdir(REPO):
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                       "https://github.com/guitorte/audio.git", REPO]).returncode:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/guitorte/audio.git", REPO], check=True)
sys.path.insert(0, f"{REPO}/vocal-lab")
import vocal_lab as V

LAB = V.start(project_name)
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except ImportError:
    gpu = None
print(f"\nGPU: {gpu or '⚠️ none. Use Runtime ▸ Change runtime type ▸ T4 GPU'}")

In [ ]:
#@title 2 · Load a song
use_drive_file = True  #@param {type:"boolean"}
drive_file_path = "/content/drive/MyDrive/audio/song.mp3"  #@param {type:"string"}
#@markdown Untick `use_drive_file` to upload a file from your computer instead.

if use_drive_file:
    source = drive_file_path
else:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    source = next(iter(uploaded))
V.load(LAB, source)

In [ ]:
#@title 3 · Analyse: tempo, key and loudness
analyse = "Original input"  #@param ["Original input", "Active file"]
#@markdown Run this on the full song (`Original input`): a bare vocal has no drums to count beats from. Module E saves its MIDI at the tempo found here.

ANALYSIS = V.analyze(LAB, analyse)

In [ ]:
#@title 🎛️ Router · choose what the next module works on (optional)
#@markdown Each module already hands its result to the next one. Use this to go back to an earlier stage, or to bring in another WAV file.
target = "Latest output"  #@param ["Latest output", "Original input", "Vocals", "Lead vocal", "De-reverbed", "Upscaled", "Denoised", "Custom path"]
custom_path = ""  #@param {type:"string"}

V.route(LAB, target, custom_path)
V.status(LAB)

In [ ]:
#@title 🧩 Module A · Separate: vocal from the song, or lead from the vocal
model = "Vocals · BS-RoFormer by viperx"  #@param ["Vocals · BS-RoFormer by viperx", "Vocals · Mel-RoFormer Big Beta 5e by unwa (fuller vocal)", "Vocals · Demucs v4 fine-tuned (also drums, bass, other)", "Lead vs backing · Mel-RoFormer Karaoke by aufr33 & viperx", "Lead vs backing · UVR MDX-Net Karaoke 2 (older)"]
#@markdown First pull the **Vocals** out of the song. To split lead from backing, run this cell again with a **Lead vs backing** model: it works on the vocal you just made.
use_result_next = True  #@param {type:"boolean"}

STEMS = V.separate(LAB, model, use_result_next)

In [ ]:
#@title 🏛️ Module B · De-reverb: remove the room
model = "De-reverb · Mel-RoFormer by anvuew"  #@param ["De-reverb · Mel-RoFormer by anvuew", "De-reverb · Mel-RoFormer by anvuew, less aggressive", "De-reverb · UVR DeEcho-DeReverb (older)"]
#@markdown Run it on a vocal, not on a full song. Try *less aggressive* if the voice turns thin.
use_result_next = True  #@param {type:"boolean"}

STEMS = V.separate(LAB, model, use_result_next)

In [ ]:
#@title 🔇 Module C · Denoise: remove hiss and background noise
model = "Denoise · Mel-RoFormer by aufr33"  #@param ["Denoise · Mel-RoFormer by aufr33", "Denoise · Mel-RoFormer by aufr33, aggressive"]
#@markdown The *aggressive* model removes more, and may take some breath and air with it.
use_result_next = True  #@param {type:"boolean"}

STEMS = V.separate(LAB, model, use_result_next)

In [ ]:
#@title ⚡ Module D · Upscale: rebuild missing highs (stereo kept)
engine = "AudioSR"  #@param ["AudioSR", "FlashSR (experimental)"]
#@markdown Run this **after** the cleaning modules: they work at 44.1 kHz and would drop the new highs. ⏱ AudioSR is slow: expect 10+ minutes per song on a T4. FlashSR takes seconds but its sound is still unjudged.
AUDIOSR_STEPS = 50  #@param {type:"slider", min:20, max:200, step:10}
CUTOFF_HZ = 0  #@param {type:"integer"}
#@markdown `0` = find where the highs were cut automatically. If the module says there is nothing to rebuild, it prints the value to put here.
use_result_next = True  #@param {type:"boolean"}

UPSCALED = V.upscale(LAB, engine, AUDIOSR_STEPS, CUTOFF_HZ, use_result_next)

In [ ]:
#@title 🎼 Module E · Vocal to MIDI
model_size = "medium"  #@param ["small", "medium", "large"]
language = ""  #@param ["", "en", "ja", "yue", "zh"]
#@markdown Works on **one sung line at a time**, so run it on a lead vocal. `language` sharpens note boundaries for those four languages; leave it empty for any other (Portuguese included).

MIDI = V.transcribe(LAB, model_size, language)

In [ ]:
#@title ✂️ Module F · Phrase chopper and sample pack
silence_threshold_db = 32  #@param {type:"slider", min:15, max:50, step:1}
min_phrase_len_sec = 0.35  #@param {type:"slider", min:0.1, max:2.0, step:0.05}
edge_padding_sec = 0.05  #@param {type:"slider", min:0.01, max:0.2, step:0.01}

PACK = V.chop(LAB, silence_threshold_db, min_phrase_len_sec, edge_padding_sec)

### Tips
- **A typical vocal chain:** A (Vocals) → A again (Lead vs backing) → B → C → D → E → F, the order of the cells. Skip what you don't need.
- **Trying another model in the same module replaces that module's earlier result.** To compare two models, copy the first result somewhere before running the second.
- **Names tell the story:** `Song_vocals_lead_dry_clean_48k.wav` went through every step, in that order.
- **Module D is the same upscaler as the Neural Audio Upscaler notebook:** only the band above the cut is new, and stereo is kept. It skips files whose highs are intact.
- **Coming back another day:** run cell 1 with the same project name, then use the Router to pick up an earlier result from Drive.
- **If a cell stops with an error**, the lines printed **above** the red traceback explain it.
- **After I push a fix**, use *Runtime ▸ Disconnect and delete runtime* before running again, or Colab keeps the old code.
- Model weights keep their own licences; several are for non-commercial use only. See the README.